# Fairness Audit: Internal Loan Approval Model
## Algorithmic Fairness Audit Playbook — Vol. 01 | Illustrative Case Study

**Author:** Hadi Noori  
**Methodology:** [Fairness Audit Playbook v3.0](../README.md)  
**Regulatory basis:** EU AI Act Art. 9/10/13 · NIST AI RMF · EEOC 29 CFR § 1607

> **Note:** This notebook uses a synthetic dataset generated to illustrate the four-phase
> audit methodology described in the Playbook (Section 03).
> It is **not** real organizational data. All names, values, and outcomes are simulated.
> For an audit on real-world data, see [`compas_audit.ipynb`](compas_audit.ipynb).

---

### Scenario

A financial institution uses an internal ML model to approve or reject loan applications.
An internal review flags a potential racial disparity in approval rates.
This audit applies the Fairness Audit Playbook to investigate, quantify, and remediate.

**Key question:** Does the model produce disparate impact against Black applicants,
and if so, is it caused by proxy variables or by legitimate creditworthiness factors?


---
## Phase 01 — Contextual Discovery (HCAT)

**Objective:** Identify proxy variables and check the 80% Rule before computing any metric.  
**Output:** Data Card


In [ ]:
import pandas as pd
import numpy as np
import sys
sys.path.append('../src')
from fairness_metrics import (
    calculate_disparities,
    bootstrap_ci,
    intersectional_scan,
    check_deployment_gates
)

df = pd.read_csv('../data/loan_synthetic.csv')

print(f"Total applicants : {len(df)}")
print(f"Race distribution:")
print(df['race'].value_counts().to_string())
print(f"\nGender distribution:")
print(df['gender'].value_counts().to_string())


### STEP 1.2 — Proxy Variable Analysis

Two features are flagged as proxies in the HCAT:

**`years_at_address`** — Residential stability correlates with race due to historical
redlining and discriminatory housing policies. White applicants average 6.5 years;
Black applicants average 3.0 years. This feature encodes structural inequality,
not individual creditworthiness.

**`voluntary_overtime`** — Hours worked beyond contract correlates with gender
due to caregiver responsibilities. Male applicants average 4.5 hrs/week;
Female applicants average 1.8 hrs/week. Using this feature penalizes caregivers.

**Action:** Both features are **removed** from the model in the remediated version.
Business Necessity analysis found that `income`, `debt_to_income`, `internal_tenure`,
and `credit_score` provide sufficient predictive power without proxy contamination.


In [ ]:
# Proxy variable analysis
print("=== Proxy Variable: years_at_address ===")
print(df.groupby('race')['years_at_address'].agg(['mean', 'median']).round(2))
print()
print("=== Proxy Variable: voluntary_overtime ===")
print(df.groupby('gender')['voluntary_overtime'].agg(['mean', 'median']).round(2))
print()
print("HCAT finding: Both features show group-level differences")
print("explained by structural inequality, not individual behavior.")
print("Action: Removed from model — no Business Necessity justification.")


### STEP 1.4 — 80% Rule Pre-Check (EEOC Legal Baseline)


In [ ]:
rate_white = df[df.race == 'White']['creditworthy'].mean()
rate_black = df[df.race == 'Black']['creditworthy'].mean()
air_labels = rate_black / rate_white

print(f"Historical approval rate — White : {rate_white:.1%}")
print(f"Historical approval rate — Black : {rate_black:.1%}")
print(f"Adverse Impact Ratio (AIR)       : {air_labels:.2f}")
print()
if air_labels < 0.80:
    print("WARNING: AIR < 0.80 — historical labels encode disparate impact.")
    print("Training any model on these labels without correction propagates bias.")
    print("Audit proceeds — remediation required at Phase 04.")


### Data Card Summary

| Field | Value |
|-------|-------|
| Dataset | Internal Loan Application — Synthetic |
| Type | Synthetic — generated to illustrate Playbook Section 03 |
| Records | 4,000 |
| Protected attributes | race, gender |
| Target label | creditworthy (ground truth) |
| Model output (biased) | model_approved_biased |
| Model output (fair) | model_approved_fair |
| Proxy flagged | years_at_address (race proxy), voluntary_overtime (gender proxy) |
| Proxy action | Removed — no Business Necessity |
| **Critical note** | **This is synthetic data. Do not use for real decisions.** |

**Data Card signed — Phase 02 may proceed.**


---
## Phase 02 — Fairness Definition Selection

### Decision Tree Walkthrough

| Question | Answer | Basis |
|----------|--------|-------|
| Are base rates equal across groups? | No | AIR_labels = 0.66 |
| Is this a punitive or assistive system? | **Assistive** | Loan approval — positive outcome |
| Is FN harm more severe than FP harm? | **Yes** | Missing a creditworthy applicant (FN) causes real financial harm |
| **Primary metric** | **Equal Opportunity (TPR Parity)** | Maximize correct approvals across groups |

### Fairness Definition Rationale

This is an **assistive system** — it grants access to financial resources.
A False Negative (denying a creditworthy applicant) causes direct economic harm
and may constitute illegal discrimination under ECOA (Equal Credit Opportunity Act).
A False Positive (approving a non-creditworthy applicant) increases default risk.

In a lending context, FN harm outweighs FP harm from a fairness perspective.
Therefore **Equal Opportunity (TPR Parity)** is the primary metric.

Note the contrast with COMPAS: same Playbook, different system type → different metric.
This is Phase 02 working as designed.

**Trade-off accepted:** An approval rate gap may persist. Documented here and in Model Card.

*Metric selected and documented before any metrics were computed.*


---
## Phase 03 — Bias Identification & Prioritization

### Bias Taxonomy

| Bias Type | Feature | Description | Priority Score |
|-----------|---------|-------------|---------------|
| Proxy Bias | years_at_address | Redlining history encoded as residential stability | Critical (60) |
| Proxy Bias | voluntary_overtime | Caregiver penalty encoded as work commitment | High (45) |
| Historical Bias | creditworthy label | Past discriminatory lending encoded in ground truth | High (40) |

### Priority Score: years_at_address
Severity = 5 (direct legal liability — ECOA) × Persistence = 4 × Difficulty = 3 = **60**  
Score > 50 is a ship-blocker requiring immediate remediation before deployment.


---
## Phase 04 — Metrics, Validation & Model Card

### Before Remediation — Biased Model


In [ ]:
metrics_biased = calculate_disparities(
    data=df,
    sensitive_attr='race',
    majority_label='White',
    minority_label='Black',
    target_label='creditworthy',
    prediction='model_approved_biased'
)

print("=== Biased Model — Disparity Metrics ===")
for k, v in metrics_biased.items():
    print(f"  {k:<35} {v}")

print()
print("=== Primary Metric: EOD (Equal Opportunity) ===")
tpr_white = df[(df.race=='White') & (df.creditworthy==1)]['model_approved_biased'].mean()
tpr_black = df[(df.race=='Black') & (df.creditworthy==1)]['model_approved_biased'].mean()
eod = tpr_white - tpr_black
print(f"  TPR White (creditworthy, approved) : {tpr_white:.1%}")
print(f"  TPR Black (creditworthy, approved) : {tpr_black:.1%}")
print(f"  EOD gap                            : {eod:.1%}")
print()
status = "FAIL" if eod > 0.05 else "PASS"
print(f"  Equal Opportunity gate: {status}")


In [ ]:
print("Running bootstrap CI...")
ci_biased = bootstrap_ci(
    data=df,
    sensitive_attr='race',
    majority_label='White',
    minority_label='Black',
    target_label='creditworthy',
    prediction='model_approved_biased',
    n_resamples=2000
)

print()
print("=== Bootstrap CI — Biased Model ===")
print(f"  EOD 95% CI       : {ci_biased['EOD_CI']}")
print(f"  EOD significant  : {ci_biased['EOD_significant']}")
print(f"  AIR 95% CI       : {ci_biased['AIR_CI']}")
print(f"  CI width         : {ci_biased['EOD_CI_width']}")


### Intersectional Scan — Race x Gender


In [ ]:
scan_biased = intersectional_scan(
    data=df,
    attr1='race',
    attr2='gender',
    target_label='creditworthy',
    prediction='model_approved_biased'
)

print("=== Intersectional Scan — Biased Model ===")
print(scan_biased.to_string(index=False))
print()
worst = scan_biased.iloc[0]
print(f"Worst-off group (lowest TPR): {worst['group']} — TPR {worst['TPR']:.1%}")
print("This group defines the audit's binding constraint.")


### Deployment Gates — Biased Model


In [ ]:
gates_biased = check_deployment_gates(metrics_biased, ci_biased)

print("=== Deployment Gates — Biased Model ===")
for gate, result in gates_biased.items():
    if gate == 'OVERALL':
        continue
    print(f"  {gate:<25} value={result['value']}  [{result['status']}]")
print()
print("=" * 55)
print(f"DECISION: {gates_biased['OVERALL']}")
print("=" * 55)


---
### After Remediation — Fair Model

**Remediation applied:** Remove proxy variables (`years_at_address`, `voluntary_overtime`).
Retrain model on legitimate creditworthiness features only.


In [ ]:
metrics_fair = calculate_disparities(
    data=df,
    sensitive_attr='race',
    majority_label='White',
    minority_label='Black',
    target_label='creditworthy',
    prediction='model_approved_fair'
)

ci_fair = bootstrap_ci(
    data=df,
    sensitive_attr='race',
    majority_label='White',
    minority_label='Black',
    target_label='creditworthy',
    prediction='model_approved_fair',
    n_resamples=2000
)

gates_fair = check_deployment_gates(metrics_fair, ci_fair)

print("=== Fair Model — Disparity Metrics ===")
print(f"  AIR  : {metrics_fair['AIR']}  (was {metrics_biased['AIR']})")
print(f"  EOD  : {metrics_fair['EOD_abs']}  (was {metrics_biased['EOD_abs']})")
print()
print("=== Deployment Gates — Fair Model ===")
for gate, result in gates_fair.items():
    if gate == 'OVERALL':
        continue
    print(f"  {gate:<25} value={result['value']}  [{result['status']}]")
print()
print("=" * 55)
print(f"DECISION: {gates_fair['OVERALL']}")
print("=" * 55)


---
## Model Card Summary

| Field | Before Remediation | After Remediation |
|-------|--------------------|-------------------|
| Model | Loan Approval v1 (biased) | Loan Approval v2 (fair) |
| Primary metric | Equal Opportunity (TPR Parity) | Equal Opportunity (TPR Parity) |
| AIR | ~0.61 | ~0.85 |
| EOD | ~0.11 | ~0.02 |
| Worst intersectional group | Black x Female | — |
| Proxy variables | years_at_address, voluntary_overtime | Removed |
| **Deployment decision** | **BLOCKED** | **DEPLOY** |

### Key Finding

Removing two proxy variables — `years_at_address` and `voluntary_overtime` —
reduced the EOD from 11% to 2% and raised the AIR from 0.61 to 0.85,
bringing all deployment gates within threshold.

This demonstrates the core value of Phase 01 (HCAT): identifying proxy variables
*before* metrics are computed allows targeted remediation rather than model replacement.

Full Model Card: [`artifacts/model_card_loan.md`](../artifacts/model_card_loan.md)
